# 03 Train and Evaluate

Trains ATLAS-IDNet plus logistic regression and random forest baselines, then writes metrics and reports.

In [1]:
import os
import sys
import zipfile
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print("Google Drive mount skipped:", exc)


def has_atlas_repo(root: Path) -> bool:
    return (root / "implementation" / "src" / "atlas_idnet" / "config.py").exists()


def limited_drive_search(base: Path, max_depth: int = 6) -> Path | None:
    if not base.exists():
        return None
    base_depth = len(base.parts)
    skip = {".git", "__pycache__", ".ipynb_checkpoints", "Dataset", "results", "work"}
    for root, dirs, files in os.walk(base):
        path = Path(root)
        depth = len(path.parts) - base_depth
        dirs[:] = [d for d in dirs if d not in skip and not d.startswith(".")]
        if depth > max_depth:
            dirs[:] = []
            continue
        if path.name == "atlas_idnet" and "config.py" in files and path.parent.name == "src":
            return (path / "config.py").parents[3]
    return None


def find_bundle() -> Path | None:
    bundle_names = ["ATLAS-IDNet_colab_bundle.zip", "atlas_idnet_colab_bundle.zip"]
    bases = [
        Path.cwd(),
        Path("/content"),
        Path("/content/drive/MyDrive/projects/ATLAS-IDNet"),
        Path("/content/drive/MyDrive/projects"),
        Path("/content/drive/MyDrive"),
    ]
    for base in bases:
        for name in bundle_names:
            candidate = base / name
            if candidate.exists():
                return candidate
    return None


def extract_bundle(bundle_path: Path) -> Path:
    target = Path("/content/ATLAS-IDNet")
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_path, "r") as archive:
        archive.extractall(target.parent)
    if has_atlas_repo(target):
        return target
    for candidate in [target, *target.parent.glob("ATLAS-IDNet*")]:
        if has_atlas_repo(candidate):
            return candidate
    raise RuntimeError(f"Extracted {bundle_path}, but implementation/src/atlas_idnet/config.py was not found.")

manual_root = os.environ.get("ATLAS_IDNET_PROJECT_ROOT")
search_candidates = []
if manual_root:
    search_candidates.append(Path(manual_root).expanduser())

search_candidates.extend([Path.cwd(), *Path.cwd().parents])
search_candidates.extend([
    Path("/content/drive/MyDrive/projects/ATLAS-IDNet"),
    Path("/content/drive/MyDrive/ATLAS-IDNet"),
    Path("/content/drive/MyDrive/onlyscholar-projects/ATLAS-IDNet"),
    Path("/content/drive/MyDrive/OnlyScholar/Projects/ATLAS-IDNet"),
    Path("/content/ATLAS-IDNet"),
    Path("/Volumes/LocalDrive1/onlyscholar-projects/ATLAS-IDNet"),
])

PROJECT_ROOT = None
bundle = find_bundle()
if bundle is not None and not manual_root:
    print("Refreshing repo bundle:", bundle)
    PROJECT_ROOT = extract_bundle(bundle)

if PROJECT_ROOT is None:
    for candidate in search_candidates:
        if has_atlas_repo(candidate):
            PROJECT_ROOT = candidate
            break

if PROJECT_ROOT is None and bundle is not None:
    print("Found repo bundle:", bundle)
    PROJECT_ROOT = extract_bundle(bundle)

if PROJECT_ROOT is None:
    for base in [
        Path("/content/drive/MyDrive/projects"),
        Path("/content/drive/MyDrive/onlyscholar-projects"),
        Path("/content/drive/MyDrive/OnlyScholar"),
        Path("/content/drive/MyDrive"),
    ]:
        PROJECT_ROOT = limited_drive_search(base)
        if PROJECT_ROOT is not None:
            break

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Could not find ATLAS-IDNet project root. The Colab runtime can run the notebook, but it cannot see the repo code. "
        "Fix: upload ATLAS-IDNet_colab_bundle.zip to /content/drive/MyDrive/projects/ATLAS-IDNet/ or set "
        "os.environ['ATLAS_IDNET_PROJECT_ROOT'] to a Drive folder that contains implementation/src/atlas_idnet/config.py."
    )

SRC_DIR = PROJECT_ROOT / "implementation" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for module_name in list(sys.modules):
    if module_name == "atlas_idnet" or module_name.startswith("atlas_idnet."):
        del sys.modules[module_name]

from atlas_idnet.config import load_config
from atlas_idnet.pipeline import create_run_dir

CONFIG_PATH = PROJECT_ROOT / "implementation" / "configs" / "default_config.json"
cfg = load_config(CONFIG_PATH)
LAST_RUN_FILE = PROJECT_ROOT / "implementation" / ".last_run_dir"

def get_or_create_run_dir(run_name="atlas_idnet_main_run"):
    if LAST_RUN_FILE.exists():
        run_dir = Path(LAST_RUN_FILE.read_text(encoding="utf-8").strip())
        run_dir.mkdir(parents=True, exist_ok=True)
        return run_dir
    run_dir = create_run_dir(cfg, run_name)
    LAST_RUN_FILE.write_text(str(run_dir), encoding="utf-8")
    return run_dir

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)
print("SRC_DIR:", SRC_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Refreshing repo bundle: /content/drive/MyDrive/projects/ATLAS-IDNet/ATLAS-IDNet_colab_bundle.zip
PROJECT_ROOT: /content/ATLAS-IDNet
CONFIG_PATH: /content/ATLAS-IDNet/implementation/configs/default_config.json
SRC_DIR: /content/ATLAS-IDNet/implementation/src


In [2]:
import torch
from atlas_idnet.pipeline import run_training_stage, build_summary_tables

run_dir = get_or_create_run_dir()
DATASET = "all"
TASK = "multiclass"
MAX_ROWS = 100000

cfg.epochs = 35
cfg.batch_size = 1024
cfg.hidden_dim = 256
cfg.dropout = 0.15
cfg.learning_rate = 8e-4
cfg.patience = 8

print("ATLAS-only training. Classical ML baselines are disabled and stale baseline folders will be removed.")
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: This is running on CPU. For target >95% CICIoT2023 results, switch Colab runtime to GPU and rerun this notebook.")

outputs = run_training_stage(
    cfg,
    run_dir,
    dataset=DATASET,
    task=TASK,
    max_rows=MAX_ROWS,
    train_baselines=False,
    train_atlas=True,
)
build_summary_tables(run_dir, task=TASK)
print("Training output root:", run_dir)
outputs


ATLAS-only training. Classical ML baselines are disabled and stale baseline folders will be removed.
CUDA available: False
2026-08-28 17:23:50,456 | INFO | atlas_idnet.training | Training models for CICIoT2023/multiclass


INFO:atlas_idnet.training:Training models for CICIoT2023/multiclass


2026-08-28 17:29:09,456 | INFO | atlas_idnet.training | Training models for NF-UQ-NIDS-v2/multiclass


INFO:atlas_idnet.training:Training models for NF-UQ-NIDS-v2/multiclass


Training output root: /content/drive/MyDrive/projects/ATLAS-IDNet/results/atlas_idnet_main_run


{'CICIoT2023': {'baselines_removed': '/content/drive/MyDrive/projects/ATLAS-IDNet/results/atlas_idnet_main_run/CICIoT2023/multiclass/models/baselines',
  'atlas': {'accuracy': 0.9503,
   'error_rate': 0.049699999999999966,
   'balanced_accuracy': 0.6512231618362871,
   'precision_macro': 0.6659982134460207,
   'precision_weighted': 0.9485932911486618,
   'precision_micro': 0.9503,
   'recall_macro': 0.6512231618362871,
   'recall_weighted': 0.9503,
   'recall_micro': 0.9503,
   'f1_macro': 0.6460417537364933,
   'f1_weighted': 0.9462302468205799,
   'f1_micro': 0.9503,
   'matthews_correlation_coefficient': 0.9457382146695468,
   'cohen_kappa': 0.945591258290881,
   'macro_detection_rate_tpr': 0.6512231618362871,
   'macro_specificity_tnr': 0.9983253031163059,
   'macro_false_alarm_rate_fpr': 0.001674696883694222,
   'macro_miss_rate_fnr': 0.34877683816371313,
   'probability_metric_error': 'not_computable',
   'model': 'ATLAS-IDNet',
   'device': 'cpu',
   'parameters': 1215013,
   'b